In [ ]:
!pip install -r requirements.txt

In [6]:
import asyncio
from azure.identity.aio import DefaultAzureCredential
from semantic_kernel.agents import AgentGroupChat, AzureAIAgent, AzureAIAgentSettings, ChatCompletionAgent
from dotenv import load_dotenv
import os
from azure.ai.projects.models import AzureAISearchTool, AzureAISearchQueryType, AsyncToolSet
from azure.ai.projects.aio import AIProjectClient

In [7]:
load_dotenv()
project_client = AzureAIAgent.create_client(credential=DefaultAzureCredential(), conn_str=os.getenv("AZURE_AI_CONNECTION_STRING"))
# project_client = AIProjectClient.from_connection_string(
#     credential=DefaultAzureCredential(),
#     conn_str=os.getenv("AZURE_AI_CONNECTION_STRING"),
# )

In [ ]:
AGENT_NAME = "PRODUCT_RECOMMENDER"
AGENT_DESCRIPTION = "An agent that provides product recommendations for outdoor trekking gear."
INSTRUCTIONS = "You are a helpful assistant that provides information about outdoor trekking gear. You do not use your internal knowledge, only use functions available to you to search for data. Provide citation for your answer."

In [9]:
connection = await project_client.connections.get(connection_name=os.environ["AI_SEARCH_CONNECTION_NAME"])
ai_search = AzureAISearchTool(
        index_connection_id=connection.id,
        index_name="webinar-ma-index",
        query_type=AzureAISearchQueryType.VECTOR_SEMANTIC_HYBRID,
        top_k=3
)
toolset = AsyncToolSet()
toolset.add(ai_search)

In [ ]:
# agent = await project_client.agents.create_agent(
#                 model="gpt-4o",
#                 name=AGENT_NAME,
#                 instructions=INSTRUCTIONS,
#                 toolset=toolset)

azure_ai_agent = AzureAIAgent(
            client=project_client,
            definition=await project_client.agents.create_agent(
                model="gpt-4o",
                name=AGENT_NAME,
                instructions=INSTRUCTIONS,
                description=AGENT_DESCRIPTION,
                toolset=toolset,
            ),
        )

In [11]:
print(f"Created agent, ID: {azure_ai_agent.id}")

Created agent, ID: asst_BFI3WFBIbnNOtSiuz0zZm5h5


In [ ]:
# thread = await project_client.agents.create_thread()

In [14]:
question = "Recommend me a good tent for outdoor, weather is rainy and windy."
thread = None
async for response in azure_ai_agent.invoke(
                    messages=question,
                    thread=thread,
                ):
                    print(f"# {response.name}: {response}")
                    thread = response.thread

# RESEARCHER: For rainy and windy outdoor conditions, I recommend considering the All-Weather Tent. This tent features:

- Waterproof fabric designed to withstand heavy rain and snow.
- A wind-resistant structure reinforced with aluminium poles for added stability during storms.
- Mesh ventilation to maintain airflow and provide bug protection.
- UV-resistant material for protection against harsh sunlight.

It is available in different sizes, accommodating 2-person, 4-person, and 6-person groups, making it versatile for couples, families, or small groups【3:0†source】.
